In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
df.to_csv("/content/drive/MyDrive/cleaned_orders.csv", index=False)
print("Saved to Google Drive! ✅")

Saved to Google Drive! ✅


In [ ]:
import pandas as pd
import numpy as np
print("Libraries importeḍ successfuly!")

Libraries importeḍ successfuly!


In [14]:
from google.colab import files
uploaded = files.upload()


Saving olist_order_reviews_dataset.csv to olist_order_reviews_dataset (1).csv
Saving olist_orders_dataset.csv to olist_orders_dataset (1).csv
Saving olist_products_dataset.csv to olist_products_dataset (1).csv
Saving olist_sellers_dataset.csv to olist_sellers_dataset (1).csv
Saving product_category_name_translation.csv to product_category_name_translation (1).csv
Saving olist_customers_dataset.csv to olist_customers_dataset (1).csv
Saving olist_geolocation_dataset.csv to olist_geolocation_dataset (1).csv
Saving olist_order_items_dataset.csv to olist_order_items_dataset (1).csv
Saving olist_order_payments_dataset.csv to olist_order_payments_dataset (1).csv


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
orders = pd.read_csv("olist_orders_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")
reviews = pd.read_csv("olist_order_reviews_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
sellers = pd.read_csv("olist_sellers_dataset.csv")
category = pd.read_csv("product_category_name_translation.csv")
geolocation = pd.read_csv("olist_geolocation_dataset.csv")

print("All files loaded successfully!")

In [ ]:
for name, df in [("orders", orders), ("customers", customers),
                 ("order_items", order_items), ("payments", payments),
                 ("reviews", reviews), ("products", products)]:
    print(f"\n{name} missing values:")
    print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

print("Date columns fixed!")

In [ ]:
orders = orders[orders['order_status'] == 'delivered']
print(f"Delivered orders: {len(orders)}")

In [ ]:
# orders + customers
df = orders.merge(customers, on='customer_id', how='left')

# + order items
df = df.merge(order_items, on='order_id', how='left')

# + payments
payments_agg = payments.groupby('order_id')['payment_value'].sum().reset_index()
df = df.merge(payments_agg, on='order_id', how='left')

# + products
df = df.merge(products, on='product_id', how='left')

# + category translation (Portuguese → English)
df = df.merge(category, on='product_category_name', how='left')

# + sellers
df = df.merge(sellers, on='seller_id', how='left')

print(f"Master table shape: {df.shape}")
df.head()

In [ ]:
# Drop rows where payment value is missing
df = df.dropna(subset=['payment_value'])

# Fill missing category names
df['product_category_name_english'] = df['product_category_name_english'].fillna('unknown')

# Calculate delivery time in days
df['delivery_days'] = (df['order_delivered_customer_date'] -
                       df['order_purchase_timestamp']).dt.days

# Extract month and year for trend analysis
df['order_month'] = df['order_purchase_timestamp'].dt.to_period('M')
df['order_year'] = df['order_purchase_timestamp'].dt.year

print("Cleaning done!")
print(df.shape)

In [ ]:
df.to_csv("cleaned_orders.csv", index=False)
print("File saved as cleaned_orders.csv")

In [ ]:
print(df.shape)
print(df.columns.tolist())
df.head()